In [1]:
#!pip install -r /Users/20s_a02/repositories/EPDE/requirements.txt


In [2]:
import numpy as np
import pandas as pd
import dill as pickle
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import epde

import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances)
import json
from epde import EpdeSearch, EpdeMultisample
import re
import itertools
from pathlib import Path
from typing import List, Dict, Tuple, Any
from data_process import DataProcessor

/opt/anaconda3/envs/Active_Matter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================
# === EQUATION DISCOVERER CLASS ===
# ==========================
class EquationDiscoverer:
    """
    Wrapper for EPDE search for discovering system equations.

    Attributes
    ----------
    samples : list
        Input data (normalized coordinates or signals).
    """
    def __init__(self, samples: list):
        self._samples = samples

    # === Discover system equations ===
    def discover_system_equation(self, epde_search_obj: EpdeSearch,
                                 additional_tokens: List, variable_names: list,dimensionality: int = 0) -> Any:
        """Performs EPDE search for system of equations."""
        factors_max_number = {'factors_num': [1, 2], 'probas': [0.85, 0.15]}
        
        trig_tokens = epde.TrigonometricTokens(freq=(0.999, 1.001), dimensionality=dimensionality)
        grid_tokens = epde.GridTokens(['t'], dimensionality=dimensionality)
        
        epde_search_obj.fit(
            data=self._samples,
            variable_names=variable_names,
            max_deriv_order=2,
            equation_terms_max_number=3,
            data_fun_pow=1,
            additional_tokens=additional_tokens+[trig_tokens, grid_tokens],
            equation_factors_max_number=factors_max_number,
            eq_sparsity_interval=(1e-6, 1)
        )

        return epde_search_obj.equations(only_print=False, only_str=False, num=4)


In [4]:
# ==========================
# === EQUATION PROCESSOR CLASS ===
# ==========================
class EquationProcessor:
    """
    Processes EPDE-discovered equations into structured tables for analysis.

    Attributes
    ----------
    regex : compiled regex
        Used to remove frequency patterns from term names.
    """

    def __init__(self):
        self.regex = re.compile(r', freq:\s\d\S\d+')

    # === Helper to update dictionaries ===
    @staticmethod
    def dict_update(d_main: Dict, term: str, coeff: float, k: int) -> Dict:
        """Updates dictionary of terms with new coefficients."""
        str_t = '_r' if '_r' in term else ''
        arr_term = re.sub('_r', '', term).split(' * ')
        perm_set = list(itertools.permutations(range(len(arr_term))))
        structure_added = False

        for p_i in perm_set:
            temp = " * ".join([arr_term[i] for i in p_i]) + str_t
            if temp in d_main:
                if k - len(d_main[temp]) >= 0:
                    d_main[temp] += [0 for _ in range(k - len(d_main[temp]))] + [coeff]
                else:
                    d_main[temp][-1] += coeff
                structure_added = True

        if not structure_added:
            d_main[term] = [0 for _ in range(k)] + [coeff]

        return d_main

   # === Convert equation to table ===
    def equation_table(self, k: int, equation, dict_main: Dict, dict_right: Dict) -> List[Dict]:
        """Creates structured dictionary from EPDE equation."""
        equation_s = equation.structure
        equation_c = equation.weights_final
        text_form_eq = self.regex.sub('', equation.text_form)

        flag = False
        for t_eq in equation_s:
            term = self.regex.sub('', t_eq.name)
            for t in range(len(equation_c)):
                c = equation_c[t]
                if f'{c} * {term} +' in text_form_eq:
                    dict_main = self.dict_update(dict_main, term, c, k)
                    equation_c = np.delete(equation_c, t)
                    break
                elif f'+ {c} =' in text_form_eq:
                    dict_main = self.dict_update(dict_main, "C", c, k)
                    equation_c = np.delete(equation_c, t)
                    break
            if f'= {term}' == text_form_eq[text_form_eq.find('='):] and not flag:
                flag = True
                dict_main = self.dict_update(dict_main, term, -1., k)

        return [dict_main, dict_right]

    # === Convert all objects to table ===
    def object_table(self, res: List, variable_names: List[str],
                     table_main: List[Dict], k: int) -> Tuple[List[Dict], int]:
        """Processes EPDE results into table format."""


        for list_SoEq in res:
            for SoEq in list_SoEq:
                for n, value in enumerate(variable_names):
                    gene = SoEq.vals.chromosome.get(value)
                    table_main[n][value] = self.equation_table(
                        k, gene.value, *table_main[n][value]
                    )
                k += 1
        return table_main, k

    # === Preprocess to DataFrame ===
    def preprocessing_table(self, variable_name: List[str],
                            table_main: List[Dict], k: int) -> pd.DataFrame:
        """Prepares final DataFrame from processed equation tables."""
        data_frame_total = pd.DataFrame()
        for dict_var in table_main:
            for var_name, list_structure in dict_var.items():
                general_dict = {}
                for structure in list_structure:
                    general_dict.update(structure)
                dict_var[var_name] = general_dict

        for dict_var in table_main:
            for var_name, general_dict in dict_var.items():
                for key, value in general_dict.items():
                    if len(value) < k:
                        general_dict[key] = value + [0. for _ in range(k - len(value))]

        data_frame_main = [{i: pd.DataFrame()} for i in variable_name]
        for n, dict_var in enumerate(table_main):
            for var_name, general_dict in dict_var.items():
                data_frame_main[n][var_name] = pd.DataFrame(general_dict)

        for n, var_name in enumerate(variable_name):
            data_frame_temp = data_frame_main[n].get(var_name).copy()
            list_columns = [f'{col}_{var_name}' for col in data_frame_temp.columns]
            data_frame_temp.columns = list_columns
            data_frame_total = pd.concat([data_frame_total, data_frame_temp], axis=1)

        return data_frame_total



In [ ]:
# ==========================
# === ROBOT PROCESSING FUNCTION ===
# ==========================
def process_robot(object_id, normalized_coord_data,pwm_signal, lev = 'micro', force = 'without', num_parts=1, 
                  output_dir=Path("EPDE_output"), save_flag:bool = True):
    """Processes robot data, splits into parts, discovers equations, and saves results."""
    print(f"\n Processing robot {object_id} with {num_parts} parts...")
    print(len(list(normalized_coord_data.values())[0]))
    cut_number = min(200, len(list(normalized_coord_data.values())[0]))
    cut_points = np.zeros((cut_number, len(object_id), 2))

    # === Directories ===
    if lev == 'micro':
        robot_dir = output_dir / f"robot_{object_id[0]}/{num_parts}_parts/{force}_force"
        robot_dir.mkdir(parents=True, exist_ok=True)

    if lev == 'meso':
        ids = '_'.join([str(i) for i in object_id])
        robot_dir = output_dir / f"robot_{ids}/{num_parts}_parts/{force}_force"
        robot_dir.mkdir(parents=True, exist_ok=True)

    for i, robot_id in enumerate(object_id):
        coords = np.array(normalized_coord_data[robot_id][:cut_number])
        cut_points[:, i, :] = coords
    

    # === Split data into parts ===
    part_size = cut_number // num_parts
    boundaries = [i * part_size for i in range(num_parts)] + [cut_number]

    for part_idx in range(num_parts):
        start_idx = boundaries[part_idx]
        end_idx = boundaries[part_idx + 1]
        part_points = cut_points[start_idx:end_idx,:,:]
        # pwm_part = np.array(pwm_signal[start_idx:end_idx])
        samples = []
        for i in range(len(object_id)):

            x = part_points[:, i, 0]
            y = part_points[:, i, 1]

            samples.append(x)
            samples.append(y)

        variable_names = []
        for i in range(len(object_id)):

            variable_names.append(f"x{i+1}")
            variable_names.append(f"y{i+1}")

        t_data = np.arange(len(part_points))
        
        print(f"\n--- Optimizing part {part_idx + 1}/{num_parts} ({start_idx}:{end_idx}) ---")

        # === Objective function for Optuna ===
        def objective(trial):
            poly_window = trial.suggest_int("poly_window", 5, 11, step=2)
            population_size = trial.suggest_int("population_size", 4, 14, step=2)
            sigma = trial.suggest_int("sigma", 0, 5, step=1)
            min_boundary = poly_window // 2 + 1
            max_boundary = 7
            boundary = trial.suggest_int("boundary", min_boundary, max_boundary)

            epde_search_obj_system = EpdeSearch(
            use_solver=False,
            boundary=boundary,
            coordinate_tensors=[t_data]
            )
            epde_search_obj_system.set_moeadd_params(population_size=population_size, training_epochs=5)
            epde_search_obj_system.set_preprocessor(
                default_preprocessor_type='poly',
                preprocessor_kwargs={'use_smoothing': True, 'sigma': sigma,'polynomial_window': poly_window}
            )
            
            '''
            external_force_token = CacheStoredTokens(
                token_type='ext_force',
                token_labels=['PWM'],
                token_tensors={'PWM': pwm_part},
                params_ranges={'power': (1, 1)},
                params_equality_ranges=None,
                meaningful=True
            )
            ''' 
            
            equation_system = EquationDiscoverer(samples)
            part_equations = equation_system.discover_system_equation(
                epde_search_obj_system,
                additional_tokens=[],
                variable_names = variable_names,
                dimensionality=0
            )

            values_per_level = [[solution.obj_fun for solution in level] for level in part_equations]
            flat_obj = [vals for level in values_per_level for vals in level]
            for level_idx, level in enumerate(values_per_level):
                print(f"Level {level_idx}:")
                for sol_idx, obj_values in enumerate(level):
                    print(f"Solution {sol_idx}: {obj_values}")

            best_score = np.inf
            n = len(variable_names)

            for obj_values in flat_obj:
                score = sum(obj_values[:n])
                constraint_sum = sum(obj_values[n:2*n])
                if constraint_sum > 0.1 * n:
                    score += 100 
                best_score = min(best_score, score)

            print(f"✅ Trial {trial} finish")
            return best_score

        # === Optuna study ===
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=10)
        best_params = study.best_params
        best_score = study.best_value
        print(f"✅ Part {part_idx + 1}: Best params {best_params}, score={best_score}")
        
        fig1 = plot_optimization_history(study)
        fig1.show()
        if save_flag:
            history_plot_path = robot_dir /f"part_{part_idx + 1}_system_history_plot.html"
            fig1.write_html(history_plot_path)
        fig2 = plot_param_importances(study)
        fig2.show()
        if save_flag:
            importances_plots_path = robot_dir /f"part_{part_idx + 1}_system_importances_plots.html"
            fig2.write_html(importances_plots_path)

        # === Save plots and params ===
        params_path = robot_dir /f"part_{part_idx + 1}_system_best_params.json"
        with open(params_path, "w") as f:
            json.dump({"best_params": best_params, "score": best_score}, f, indent=4)

        # === Final EPDE system with best params ===
        best_boundary = best_params["boundary"]
        best_window = best_params["poly_window"]
        best_sigma = best_params["sigma"]
        best_population_size = best_params["population_size"]

        epde_search_obj_system = EpdeSearch(
            use_solver=False,
            boundary=best_boundary,
            coordinate_tensors=[t_data]
        )
        epde_search_obj_system.set_moeadd_params(population_size=best_population_size, training_epochs=10)
        epde_search_obj_system.set_preprocessor(
            default_preprocessor_type='poly',
            preprocessor_kwargs={'use_smoothing': True, 'sigma': best_sigma,'polynomial_window': best_window}
        )
        equation_system = EquationDiscoverer(samples)
        part_equations = equation_system.discover_system_equation(
            epde_search_obj_system,
            additional_tokens=[],
            variable_names = variable_names,
            dimensionality=0
        )

        '''
        external_force_token = CacheStoredTokens(
            token_type='ext_force',
            token_labels=['PWM'],
            token_tensors={'PWM': pwm_part},
            params_ranges={'power': (1, 1)},
            params_equality_ranges=None,
            meaningful=True
        )
        pwm_derivs_token = ExternalDerivativesTokens(
            'PWM_derivs',                     # имя токена
            boundary=4,                       # как и у других данных
            time_axis=0,                      # ось времени (0, т.к. одномерная зависимость)
            base_token_label='PWM',           # имя базового токена
            token_tensor=pwm_signal,          # сам сигнал
            max_orders=2,                     # до какой производной считать (1 -> dPWM/dt, 2 -> d²PWM/dt²)
            deriv_method='poly',              # метод вычисления производных
            deriv_method_kwargs={
                'smooth': True,
                'grid': [t_data]
            },
            params_ranges={'power': (1, 1)},  # оставляем без возведения в степень
            params_equality_ranges=None,
            meaningful=True
        )
        '''
        
        values_per_level = [
        [solution.obj_fun for solution in level]
                            for level in part_equations]
        for level_idx, level in enumerate(values_per_level):
            print(f"Level {level_idx}:")
            for sol_idx, obj_values in enumerate(level):
                print(f"Solution {sol_idx}: {obj_values}")
        for level_idx, level in enumerate(part_equations):
            print(f"Level {level_idx}:")
            for sol_idx, solution in enumerate(level):
                print(f"  Solution {sol_idx}:")
                for eq in solution.vals:
                    print("    ", eq.text_form)
                for eq in solution.vals:
                    print("    ", eq.latex_form)
        
        if save_flag:
            output_file = robot_dir /f"part_{part_idx + 1}_obj_func_and_equations.txt"

            with open(output_file, "w") as f:
                # Сначала выводим значения obj_fun
                values_per_level = [
                    [solution.obj_fun for solution in level]
                    for level in part_equations
                ]
                for level_idx, level in enumerate(values_per_level):
                    f.write(f"Level {level_idx}:\n")
                    for sol_idx, obj_values in enumerate(level):
                        f.write(f"Solution {sol_idx}: {obj_values}\n")
                
                # Теперь выводим сами уравнения
                for level_idx, level in enumerate(part_equations):
                    f.write(f"\nLevel {level_idx}:\n")
                    for sol_idx, solution in enumerate(level):
                        f.write(f"  Solution {sol_idx}:\n")
                        # Если у тебя solution.vals существует (EPDE объект)
                        if hasattr(solution, "vals"):
                            for eq in solution.vals:
                                f.write(f"    {eq.text_form}\n")
                            for eq in solution.vals:
                                f.write(f"    {eq.latex_form}\n")
                        else:
                            # Если solution это просто список строк или numpy array
                            for eq in solution:
                                f.write(f"    {eq}\n")

            print(f"Output saved to {output_file}")

        # === Process equations into table ===
        equation_processor = EquationProcessor()
        variable_names = variable_names
        table_main = [{i: [{}, {}]} for i in variable_names]
        k = 0
        vals_flat = []
        for elem in part_equations:
            vals_flat.extend(elem)

        table_main, k = equation_processor.object_table(
            [vals_flat], variable_names, table_main, k
        )
        print(f"\nSystem of DE {part_idx + 1} (startpoint {start_idx}-{end_idx - 1}):")
        for i, eq in enumerate(part_equations):
            print(f"EQ {i + 1}:")
            print(eq)

        # === Save table ===
        if save_flag:
            frame_main = equation_processor.preprocessing_table(variable_names, table_main, k)
            output_path = robot_dir /f"system_{part_idx + 1}.csv"
            frame_main.to_csv(output_path, sep=',', encoding='utf-8')
            print(f"Saved: {output_path}")
    
# ==========================
# === MAIN FUNCTION ===
# ==========================
def main(raw_data, target_robots:List = None, robots_form: str = 'circle' , num_parts:int =1, lev:str  = 'micro', 
         particle_flag:bool =False, save_flag:bool = True):
    processor = DataProcessor(raw_data)
    processor.extract_data()
    pwm_signal=None
    #pwm_signal = np.array(processor.angle_data[target_robot])
    normalized_coord_data = processor.normalize_all_coordinates()
    robots_to_process = []

    if (particle_flag) and (robots_form == 'circle'):
        for target_robot in target_robots:
            particle_data = dict(np.load(f"{Path(robots_form)}/particle_coordinates.npz"))
            particle_coords = np.array(particle_data[f'{target_robot}'][:200])
            x_particle, y_particle = particle_coords[:, 0], particle_coords[:, 1]
            normalized_coord_data, key = processor.transform_new_robot(x_particle, y_particle, target_robot+100)
            robots_to_process += [key]
    else:
        robots_to_process = target_robots    
    

    if lev == 'micro':
        output_dir = Path(robots_form)/"EPDE_output_micro"

    if lev == 'meso':
        output_dir = Path(robots_form)/"EPDE_output_mezo"
        
    process_robot(robots_to_process, normalized_coord_data, pwm_signal,num_parts=num_parts,lev = lev, output_dir=output_dir,
                  save_flag = save_flag)

robots_form = 'oval' #'oval' or 'circle'
if robots_form == 'oval':
    with open('oval/data/oval_data_[30_bots_PWM_1_exp_1].pickle', 'rb') as file: 
        raw_data = pickle.load(file)
elif robots_form == 'circle':
    with open('circle/data/circle_data_00_330_[30_bots_PWM_10_15cw_15ccw_D_41cm].MP4.pickle', 'rb') as file: 
        raw_data = pickle.load(file)

level = 'meso_cluster' #'micro', 'meso_cluster', 'meso_union'
if level == 'micro':
    lev = 'micro'
elif level in ['meso_cluster', 'meso_union']:
    lev = 'meso'

path_levels_robots_ids = f'{Path(robots_form)}/levels_robots_ids.json'
with open(path_levels_robots_ids, "r") as f:
    levels_robots_ids = json.load(f)

particle_flag = False

save_flag = True

for target_robots in levels_robots_ids[level]:
    main(raw_data, target_robots=target_robots,robots_form=robots_form, num_parts=1, lev = lev, 
         particle_flag=particle_flag, save_flag = save_flag)
# trials, traning_epoch
